In [1]:
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed
import os

In [2]:
commands = []

# for itr in range(3):
#     for model in ['hybrid', 'T_rope', 'mamba']:
#         cmd = f"python3 main.py --model {model} --train_task decode-recall --eval_task decode-recall --sequence_length 200 --eval_equence_length 200 --lr 0.0003 --epochs 1 --save False --run_number {itr}"
#         cmd += " --min_train_length 10 --max_train_length 50 --min_eval_length 10 --max_eval_length 200 --eval_jump_type linear --eval_linear_jump_size 10 --p 0.2 --eval_p 0.3"
#         commands.append(cmd)

# for eval_p in [0.01, 0.05, 0.1, 0.3, 0.5, 0.8, 0.9]:
# for eval_p in [0.01, 0.9]:
#     for itr in range(3):
#         for model in ['hybrid', 'T_rope', 'mamba']:
#             cmd = f"python3 main.py --model {model} --train_task decode-recall --eval_task decode-recall --sequence_length 200 --eval_equence_length 200 --lr 0.0003 --epochs 1 --save False --run_number {itr}"
#             cmd += " --min_train_length 10 --max_train_length 50 --min_eval_length 10 --max_eval_length 200 --eval_jump_type linear --eval_linear_jump_size 10"
#             cmd += f" --p 0.2 --eval_p {eval_p}"
#             commands.append(cmd)

for eval_p in [0.01, 0.05, 0.1, 0.3, 0.5, 0.8, 0.9]:
    for itr in range(3): # 3
        for model in ['hybrid', 'T_rope', 'mamba']:
            cmd = f"python3 main.py --model {model} --train_task decode-recall --eval_task decode-recall --sequence_length 200 --eval_equence_length 200 --lr 0.0003 --epochs 1 --save False --run_number {itr}"
            cmd += " --min_train_length 10 --max_train_length 50 --min_eval_length 10 --max_eval_length 200 --eval_jump_type linear --eval_linear_jump_size 10"
            # cmd += f" --p 0.2 --eval_p {eval_p}"
            cmd += f" --p {eval_p} --eval_p 0.2"
            commands.append(cmd)

In [3]:
commands[:10]

['python3 main.py --model hybrid --train_task decode-recall --eval_task decode-recall --sequence_length 200 --eval_equence_length 200 --lr 0.0003 --epochs 1 --save False --run_number 0 --min_train_length 10 --max_train_length 50 --min_eval_length 10 --max_eval_length 200 --eval_jump_type linear --eval_linear_jump_size 10 --p 0.01 --eval_p 0.2',
 'python3 main.py --model T_rope --train_task decode-recall --eval_task decode-recall --sequence_length 200 --eval_equence_length 200 --lr 0.0003 --epochs 1 --save False --run_number 0 --min_train_length 10 --max_train_length 50 --min_eval_length 10 --max_eval_length 200 --eval_jump_type linear --eval_linear_jump_size 10 --p 0.01 --eval_p 0.2',
 'python3 main.py --model mamba --train_task decode-recall --eval_task decode-recall --sequence_length 200 --eval_equence_length 200 --lr 0.0003 --epochs 1 --save False --run_number 0 --min_train_length 10 --max_train_length 50 --min_eval_length 10 --max_eval_length 200 --eval_jump_type linear --eval_line

In [4]:
def run_command(cmd):
    """Run a single shell command and return (cmd, returncode, stdout, stderr)."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return cmd, result.returncode, result.stdout, result.stderr

max_workers = 1 # 5

results = ""

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = {executor.submit(run_command, cmd): cmd for cmd in commands}

    for future in as_completed(futures):
        cmd, returncode, stdout, stderr = future.result()

        if returncode != 0:
            print(f"[{cmd}] exited with {returncode}")
        
        if returncode == 1:
            executor.shutdown()
            print(stdout)
            print(stderr)
            assert False

        results += "<split>Command: " + cmd + "\n" + stdout + "\n"

with open("results/exp.txt", "w") as outfile:
    outfile.write(results)